Banco de dados a ser usado

[Detecção de fraude em cartão de crédito](https://drive.google.com/file/d/1oKM8g7l19t_puV2nQ3rSSXxrQeYDg7t8/view?usp=drive_link)


Fazer uma pipeline completa de otimização:

1. Separar em treino (70%), teste (20%) e validação (10%).
2. Fazer o tunning de hiperparametro para os modelos:

  a. Random Forest.
  * max_depth: uniforme de 5 a 10
  * n_estimators: uniforme de 100 a 500
  
  b. Xgboost
  * learning_rate: exponencial 0.1
  * max_depth: uniforme de 5 a 10
  * n_estimators: uniformde de 5 a 10
  
  c. SVM
  * C: uniforme de 1 a 1000
  * gamma: uniforme de .01 a 1
  
  d. Lasso
  * Alpha: 0.1 a 10
  
  e. Ridge
  * Alpha: 0.1 a 10

Todos executar 50 vezes.
3. Treinar cada um dos modelos com a melhor configuração
4. Definir o ponto de corte utilizando o banco de validação e como métrica o f1-score da classe minoritaria
5. Avaliar o classificador no banco de teste

In [16]:
import pandas as pd
import numpy as np
import scipy

In [4]:
data = pd.read_csv("creditcard_sample.csv") 

In [5]:
Y = data.Class
X = data.drop(columns=["Class"], axis=1)

In [6]:
from sklearn.model_selection import train_test_split

<frozen importlib._bootstrap>:228: RuntimeWarning: scipy._lib.messagestream.MessageStream size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


In [7]:
X_trainvalid, X_test, Y_trainvalid, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_trainvalid, Y_trainvalid, test_size=1 / 8, random_state=42)

print(np.mean(Y_valid))

0.08863636363636364


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [9]:
model_rf = RandomForestClassifier()

In [17]:
params = {
    "max_depth": scipy.stats.randint(5, 10),
    "n_estimators": scipy.stats.randint(100, 500),
}

In [23]:
clf = RandomizedSearchCV(model_rf, params, scoring="f1_macro", n_iter=20)

In [24]:
clf.fit(X_train, Y_train)

RandomizedSearchCV(estimator=RandomForestClassifier(), n_iter=20,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026C0AE3FE80>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000026C017A3910>},
                   scoring='f1_macro')

In [27]:
clf.best_score_

0.9584262327703499

In [28]:
pd.DataFrame(clf.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,1.144864,0.184146,0.030184,0.005596,5,165,"{'max_depth': 5, 'n_estimators': 165}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
1,3.106551,0.429971,0.069076,0.008811,5,414,"{'max_depth': 5, 'n_estimators': 414}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
2,0.907093,0.099505,0.020389,0.002655,8,105,"{'max_depth': 8, 'n_estimators': 105}",0.957244,0.962919,0.951467,0.946556,0.969034,0.957444,0.007983,17
3,2.885352,0.141941,0.069652,0.007178,6,407,"{'max_depth': 6, 'n_estimators': 407}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
4,1.379551,0.033399,0.040180,0.010100,6,201,"{'max_depth': 6, 'n_estimators': 201}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
5,1.732332,0.202218,0.036180,0.003763,9,189,"{'max_depth': 9, 'n_estimators': 189}",0.957244,0.962919,0.951467,0.946556,0.969034,0.957444,0.007983,17
6,3.418247,0.056391,0.068567,0.002238,8,422,"{'max_depth': 8, 'n_estimators': 422}",0.957244,0.962919,0.951467,0.946556,0.969034,0.957444,0.007983,17
7,1.040037,0.099434,0.028081,0.003285,5,168,"{'max_depth': 5, 'n_estimators': 168}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
8,3.582429,0.076175,0.079803,0.011819,7,477,"{'max_depth': 7, 'n_estimators': 477}",0.957244,0.962919,0.951467,0.951467,0.969034,0.958426,0.006797,1
9,3.431249,0.354289,0.090194,0.015736,7,429,"{'max_depth': 7, 'n_estimators': 429}",0.957244,0.962919,0.951467,0.946556,0.969034,0.957444,0.007983,17


In [30]:
from xgboost import XGBClassifier

In [31]:
model_xgb = XGBClassifier()